# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates step-by-step exploration of the FAIR² dataset using the `mlcroissant` library, starting from metadata inspection, through data extraction, to initial analysis and visualization.

### Dataset Source
This dataset is defined by a Croissant schema accessible at the following URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading
We load metadata and records from the dataset using `mlcroissant`. This will give us access to the dataset structure and records, including fields and record set identifiers.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Display core metadata
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Date published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
We'll explore all available record sets, their `@id`s, and the fields (columns) they supply. All references to entities in the dataset use their unique `@id`.

In [ ]:
# List all record sets and their fields using their `@id`
print("Available record sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    field_ids = [field.id for field in rs.fields]
    print(f"  Fields (@id): {field_ids}\n")

# For demonstration, list the first 2 records of the first record set (if available):
if record_sets:
    rs0_id = record_sets[0].id
    print(f"First 2 records from record set {rs0_id}:")
    for i, record in enumerate(dataset.records(record_set=rs0_id)):
        print(record)
        if i==1:
            break

## 3. Data Extraction
Now we'll extract full tabular data from each available record set into pandas DataFrames, using their `@id`. We'll inspect the columns present in the primary dataset.

In [ ]:
# Collect data from all record sets into DataFrames
all_record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

# Pull all records for each set
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Inspect columns and first few records of the first main record set
if all_record_set_ids:
    main_rs_id = all_record_set_ids[0]
    print(f"Columns in record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform some simple EDA. We'll select a numeric field using its exact `@id` (e.g., patient age), filter records, normalize the numeric field, and optionally group by a categorical field. All fields below are referenced by their `@id` as per the data model.

In [ ]:
# Choose appropriate field @id based on record set fields (see earlier data overview cell)
# These are examples - please verify actual field @ids and names that appear in your schema:
main_rs_id = all_record_set_ids[0]
df = dataframes[main_rs_id]
print(f"Columns in {main_rs_id}: {df.columns.tolist()}")

# Find numeric fields (e.g., age). Replace with the actual ID from your overview cell:
numeric_field_id = None
for col in df.columns:
    if 'age' in col.lower() or 'interval' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    numeric_field_id = df.select_dtypes('number').columns[0] if not df.select_dtypes('number').empty else df.columns[0]  # fallback

print(f"Selected numeric field for EDA: {numeric_field_id}")

# Filter for age or interval > threshold (e.g., 50); adjust threshold as appropriate
threshold = 50
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold]
print(f"Records where {numeric_field_id} > {threshold} (showing up to 5):")
display(filtered_df.head())

# Normalize the selected numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()

print(f"Normalized {numeric_field_id} for filtered sample (showing up to 5):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a grouping field (e.g., sex, cancer_type) by @id from the available columns
group_field_id = None
for col in df.columns:
    if any(kw in col.lower() for kw in ['sex', 'gender', 'cancer', 'msi']):
        group_field_id = col
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Let's plot the distribution of the selected numeric field (e.g., age or diagnosis interval), and compare distributions across a grouping field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Compare group distributions if grouping field is present
if group_field_id:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

- This notebook demonstrated loading the FAIR² Croissant dataset using its schema URL and exploring its record sets and fields using their `@id`s.
- We extracted tabular data, referenced columns by their `@id`, performed basic filtering and normalization, and visualized numeric field distributions.
- All dataset elements were programmatically referenced and manipulated respecting explicit schema definitions and field identifiers.
- For further machine learning applications, this notebook provides a reproducible foundation to access, process, and analyze Croissant-standardized biomedical datasets.